<a href="https://colab.research.google.com/github/imothep66/chat/blob/main/clone%20vozXTTS_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dataset building + XTTS finetuning and inference

#### Running the demo
To start the demo run the first two cells (ignore pip install errors in the first one)

Then click on the link `Running on public URL: ` when the demo is ready.

#### Downloading the results

You can run cell [3] to zip and download default dataset path

You can run cell [4] to zip and download the latest model you trained

### Installing the requirements

In [ ]:
!pip install -q uv
!uv venv /opt/xtts --python 3.10
!uv pip install --python /opt/xtts/bin/python coqui-tts faster-whisper gradio==4.44.1


In [ ]:
%cd /content/TTS
!MPLBACKEND=Agg /opt/xtts/bin/python TTS/demos/xtts_ft_demo/xtts_demo.py --share


In [ ]:
!uv pip install --python /opt/xtts/bin/python "huggingface_hub<0.26.0" "starlette==0.38.6" "fastapi==0.112.4"
%cd /content/TTS
!/opt/xtts/bin/python TTS/demos/xtts_ft_demo/xtts_demo.py --share


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"

!uv pip install --python /opt/xtts/bin/python torch torchaudio --index-url https://pytorch.org
!uv pip install --python /opt/xtts/bin/python coqui-tts faster-whisper "gradio==4.44.1" "huggingface_hub<0.26.0" "starlette==0.38.6" "fastapi==0.112.4"

%cd /content/TTS
!/opt/xtts/bin/python TTS/demos/xtts_ft_demo/xtts_demo.py --share


In [12]:
import os
os.environ["MPLBACKEND"] = "Agg"
os.environ["COQUI_TOS_AGREED"] = "1"

!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!uv pip install --python /opt/xtts/bin/python -q "torchaudio<2.6.0" "torch<2.6.0" "transformers==4.45.2" "faster-whisper==1.0.3" "gradio==4.44.1" "pydantic==2.9.2" "pydantic-core==2.23.4" "huggingface_hub<0.26.0" "starlette==0.38.6" "fastapi==0.112.4" soundfile librosa

!sed -i 's/total=content_duration/total=int(content_duration)/g' /opt/xtts/lib/python3.10/site-packages/faster_whisper/transcribe.py
!sed -i 's/weights_only=None/weights_only=False/g' /opt/xtts/lib/python3.10/site-packages/TTS/utils/io.py 2>/dev/null || true

%cd /content/TTS
!/opt/xtts/bin/python TTS/demos/xtts_ft_demo/xtts_demo.py


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/content/TTS

2026-11-12 04:12:26,517 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
Running on local URL:  http://0.0.0.0:5003
2026-11-12 04:12:26,639 [INFO] HTTP Request: GET http://localhost:5003/startup-events "HTTP/1.1 200 OK"
2026-11-12 04:12:26,670 [INFO] HTTP Request: HEAD http://localhost:5003/ "HTTP/1.1 200 OK"
2026-11-12 04:12:26,673 [INFO] HTTP Request: GET https://checkip.amazonaws.com/ "HTTP/1.1 200 "
2026-11-12 04:12:26,732 [INFO] HTTP Request: GET https://api.gradio.app/v2/tunnel-request "HTTP/1.1 200 OK"
Running on public URL: https://a70e612ea2347992b0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
Loading Whisper Model!

piper onnx


In [15]:
import base64, subprocess, os

cmd = base64.b64decode("Z2l0IGNsb25lIC1xIGh0dHBzOi8vZ2l0aHViLmNvbS9yaGFzc3B5L3BpcGVyLmdpdA==").decode()
os.chdir("/content")
subprocess.run(cmd, shell=True)

os.chdir("/content/piper/src/python")
subprocess.run("pip install -q -r requirements.txt", shell=True)
subprocess.run("pip install -q cython librosa==0.9.2 onnxruntime", shell=True)
subprocess.run("bash build_monotonic_align.sh", shell=True)

os.makedirs("/content/piper_treino", exist_ok=True)
os.chdir("/content/piper_treino")
dl = base64.b64decode("d2dldCAtcSBodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RhdGFzZXRzL3JoYXNzcHkvcGlwZXItY2hlY2twb2ludHMvcmVzb2x2ZS9tYWluL3B0L3B0X0JSL2ZhYmVyL21lZGl1bS9lcG9jaCUzRDQ2Njktc3RlcCUzRDEzNzE1OTAuY2twdCAtTyBiYXNlX3B0YnIuY2twdA==").decode()
subprocess.run(dl, shell=True)
print("PASSO 1 CONCLUIDO!")


PASSO 1 CONCLUIDO!


In [ ]:
import os, subprocess

os.makedirs("/content/piper_treino", exist_ok=True)
if os.path.exists("/content/piper_treino/base_ptbr.ckpt"):
    os.remove("/content/piper_treino/base_ptbr.ckpt")

p1 = "https://huggingface.co"
p2 = "/datasets/rhasspy/piper-checkpoints"
p3 = "/resolve/main/pt/pt_BR/faber/medium"
p4 = "/epoch=6159-step=1230728.ckpt"
url = p1 + p2 + p3 + p4

subprocess.run(["curl", "-L", url, "-o", "/content/piper_treino/base_ptbr.ckpt"])

os.chdir("/content/piper/src/python")
subprocess.run([
    "/opt/xtts/bin/python", "-m", "piper_train",
    "--dataset-dir", "/content/piper_treino/treino",
    "--accelerator", "gpu",
    "--devices", "1",
    "--batch-size", "16",
    "--validation-split", "0.05",
    "--num-test-examples", "0",
    "--max_epochs", "6959",
    "--resume_from_checkpoint", "/content/piper_treino/base_ptbr.ckpt",
    "--checkpoint-epochs", "100",
    "--precision", "32"
])


In [25]:
%cd /content/piper/src/python
!/opt/xtts/bin/python -m piper_train \
  --dataset-dir /content/piper_treino/treino \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 0 \
  --max_epochs 6959 \
  --resume_from_checkpoint /content/piper_treino/base_ptbr.ckpt \
  --checkpoint-epochs 100 \
  --precision 32


/content/piper/src/python
DEBUG:piper_train:Namespace(dataset_dir='/content/piper_treino/treino', checkpoint_epochs=100, quality='medium', resume_from_single_speaker_checkpoint=None, logger=True, enable_checkpointing=True, default_root_dir=None, gradient_clip_val=None, gradient_clip_algorithm=None, num_nodes=1, num_processes=None, devices='1', gpus=None, auto_select_gpus=None, tpu_cores=None, ipus=None, enable_progress_bar=True, overfit_batches=0.0, track_grad_norm=-1, check_val_every_n_epoch=1, fast_dev_run=False, accumulate_grad_batches=None, max_epochs=6959, min_epochs=None, max_steps=-1, min_steps=None, max_time=None, limit_train_batches=None, limit_val_batches=None, limit_test_batches=None, limit_predict_batches=None, val_check_interval=None, log_every_n_steps=50, accelerator='gpu', strategy=None, sync_batchnorm=False, precision=32, enable_model_summary=True, num_sanity_val_steps=2, resume_from_checkpoint='/content/piper_treino/base_ptbr.ckpt', profiler=None, benchmark=None, reloa

In [26]:
import glob
%cd /content/piper/src/python

ckpts = sorted(glob.glob("/content/piper_treino/treino/lightning_logs/*/checkpoints/*.ckpt"))
ultimo_ckpt = ckpts[-1]

!/opt/xtts/bin/python -m piper_train.export_onnx "{ultimo_ckpt}" /content/ultron.onnx
!cp /content/piper_treino/treino/config.json /content/ultron.onnx.json
print("PRONTO PARA BAIXAR!")


/content/piper/src/python
/opt/xtts/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f

In [27]:
import glob
%cd /content/piper/src/python
!uv pip install --python /opt/xtts/bin/python -q onnx

ckpts = sorted(glob.glob("/content/piper_treino/treino/lightning_logs/*/checkpoints/*.ckpt"))
ultimo_ckpt = ckpts[-1]

!/opt/xtts/bin/python -m piper_train.export_onnx "{ultimo_ckpt}" /content/ultron.onnx
!cp /content/piper_treino/treino/config.json /content/ultron.onnx.json
print("EXPORTACAO CONCLUIDA COM SUCESSO!")


/content/piper/src/python
/opt/xtts/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f

### Running the gradio UI

In [4]:
!python TTS/TTS/demos/xtts_ft_demo/xtts_demo.py --batch_size 2 --num_epochs 6

python3: can't open file '/content/TTS/TTS/TTS/demos/xtts_ft_demo/xtts_demo.py': [Errno 2] No such file or directory


### Downloading the dataset

In [5]:
from google.colab import files

!zip -q -r dataset.zip /tmp/xtts_ft/dataset
files.download('dataset.zip')


zip error: Nothing to do! (try: zip -q -r dataset.zip . -i /tmp/xtts_ft/dataset)


FileNotFoundError: Cannot find file: dataset.zip

### Downloading the model

In [ ]:
from google.colab import files
import os
import glob
import torch

def find_latest_best_model(folder_path):
    search_path = os.path.join(folder_path, '**', 'best_model.pth')
    files = glob.glob(search_path, recursive=True)
    latest_file = max(files, key=os.path.getctime, default=None)
    return latest_file

model_path = find_latest_best_model("/tmp/xtts_ft/run/training/")
checkpoint = torch.load(model_path, map_location=torch.device("cpu"))
del checkpoint["optimizer"]
for key in list(checkpoint["model"].keys()):
    if "dvae" in key:
        del checkpoint["model"][key]
torch.save(checkpoint, "model.pth")
model_dir = os.path.dirname(model_path)
files.download(os.path.join(model_dir, 'config.json'))
files.download(os.path.join(model_dir, 'vocab.json'))
files.download('model.pth')